### This notebook provides an end-to-end example of how to extract a single HETG spectrum from a crowded region and run CrissCross to handle confusion. Much of this tutorial will be turned into a CIAO page describing CrissCross and its parameters.

In [ ]:
from ciao_contrib.runtool import *
from sherpa.astro.ui import *
import numpy as np
import matplotlib.pyplot as plt
import os

### execute the criss_cross.py script to access functions (for now)

In [ ]:
run criss_cross.py

### Download an HETG observation for the tutorial. I recommend a crowded field like the Orion Nebula Cluster so here we download obsID 3

In [ ]:
os.system('download_chandra_obsid 3')

### Lets select a source in the field of view and extract its spectrum with chandra_repro using the 'tg_zo_position' parameter.

In [ ]:
#SRC NAME -- COUP 394
#83.79475306,-5.39575306,393

coup_num = 394
src_RA = 83.79475306
src_DEC = -5.39575306
obs = 3
coup_root = f'COUP_{coup_num}'

In [ ]:
#run chandra repro using the zeroth order source location for COUP 689

chandra_repro.punlearn()
chandra_repro.indir = obs
chandra_repro.root = coup_root
chandra_repro.tg_zo_position = f'{src_RA},{src_DEC}'
chandra_repro.clobber=False

print(chandra_repro)

a = chandra_repro()

### Lets take a look at the source's spectrum in Sherpa

In [ ]:
pha_data = f'{obs}/repro/{coup_root}_repro_pha2.fits'

clean() #run clean in case 

load_pha(1, pha_data)

### Sherpa doesn't yet automatically retrieve all the response files when loading an HETG PHA2 file. Lets use CrissCross functions find_resp_files and match_resp_order to facilitate ARF and RMF loading.

In [ ]:
help(find_resp_files)

arf_list = find_resp_files(pha_data, 'arf', resp_dir_par=None)
rmf_list = find_resp_files(pha_data, 'rmf', resp_dir_par=None)

print(f'ARF Files: {arf_list}')
print(f'RMF Files: {rmf_list}')

### find_resp_match attempts to identify the response files and match_resp_order will make sure the identified responses are in the same order as the PHA2 spectra loaded into sherpa with load_data.

In [ ]:
help(match_resp_order)

arf_list_ordered = match_resp_order(pha_data, arf_list, 'arf')
rmf_list_ordered = match_resp_order(pha_data, rmf_list, 'rmf')

print(f'ARF Files: {arf_list_ordered}')
print()
print(f'RMF Files: {rmf_list_ordered}')


### ARFs and RMFs have been identified and put in the same order as the PHA2 file. Now we can load them and delete the orders that were not generated via chandra_repro (-2,-3,+2,+3). 

In [ ]:
for i in range(0,12):
    if rmf_list_ordered[i] != 'no match':
        load_arf(id=i+1, arg=arf_list_ordered[i])
        load_rmf(id=i+1, arg=rmf_list_ordered[i])
    else:
        delete_data(i+1) #must delete PHA2 datasets that don't have responses because otherwise one can't plot in wavelength/energy.

### The spectra and responses are loaded so here we will see what the uncleaned (confused) spectra look like

### We can bin and view all the spectra at once

In [ ]:
#to view all four orders at once (heg +1/-1 and meg+1/-1)
spec_ids = [3,4,9,10]

set_analysis('wave') #view spectra in wavelength

bin_counts=3
for i in spec_ids:
    group_counts(i, bin_counts)

plot("data", 3, "data", 4, "data", 9, "data", 10)

In [ ]:
#we can also plot the ARFs for each order
plot("arf", 3, "arf", 4, "arf", 9, "arf", 10)

### To facilitate a comparison of cleaned vs uncleaned spectra, we'll save the ungrouped spectra data to python variables for later plotting

In [ ]:
spec_arr = [None]*4
spec_ids = [3,4,9,10]
hetg_info = ['heg-1', 'heg+1', 'meg-1', 'meg+1']

for i in range(len(spec_arr)):
    spec_arr[i] = get_data(id=spec_ids[i])

# Create a 2×2 figure (4 sub‑plots)
fig, axes = plt.subplots(
    nrows=2, ncols=2,                # 2 rows, 2 columns
    figsize=(10, 8),                # optional: figure size in inches
    sharex=True, sharey=True,      # share axes (optional)
)

ax = axes.ravel()

for i in range(len(spec_arr)):
    ax[i].plot(spec_arr[i].x, spec_arr[i].y)
    ax[i].set_xlim(0,30)
    ax[i].set_xlabel('Wavelength')
    ax[i].set_ylabel('Counts s$^{-1}$ $A^{-1}$')
    ax[i].set_title(f'{coup_root} {hetg_info[i]}')

fig.tight_layout(pad=1.0)
plt.show()

### To identify potential confusion we'll run CrissCross on the observation. This requires an input list of all X-ray sources potentially in the field of view. I have included such a list for this tutorial but users will have to produce their own. Most of the Chandra HETG observations with more than a few sources in the field of view have been previously observed with Chandra imaging. Running wavdetect on an archival non-HETG imaging observaiton is a good way to produce an easy source list. Otherwise, users will have to use another wavelength or identify the 0th order sources in an HETG observation by eye and manually create a list.

In [ ]:
help(run_crisscross)

evt2_path = f'{obs}/repro/{coup_root}_repro_evt2.fits'

run_crisscross(cc_outdir='cc_tutorial', arf_ratios_dir='input_files', main_list='input_files/full_coup_src_list.tsv', subset_src_list = None, clean_single_RA = src_RA, clean_single_DEC = src_DEC, clean_single_root=coup_root, evt2_file=evt2_path, wavdetect_file=None, clobber_par=True, min_arm_counts=160)

### CrissCross has finished and produced the relevant cleaning table. We can view the identified confusion for each arm with dmlist (or any fits table reader)

In [ ]:
cc_table_coup = f'cc_tutorial/output_dir_obsid_3/confusion_output_files/table_fits_data/confused_{coup_root}_consolidated_obsID_3.fits'

dmlist.punlearn
dmlist.infile = f'{cc_table_coup}[flag=confused, order=-1,+1, grating_type=meg]'
dmlist.opt='data'
dmlist.rows='1:50'

dmlist()

In [ ]:
dmlist.punlearn
dmlist.infile = f'{cc_table_coup}[flag=confused, order=-1,+1, grating_type=heg]'
dmlist.opt='data'
dmlist.rows='1:50'

dmlist()

### We can see that CrissCross identified some confusion (flag column 'confused'). The 'wave_low' and 'wave_high' columns indicate the wavelength range recommended to be ignored due to the liklihood of HETG confusion by source 'confuser_srcid'. The 'confusion_type' column shows which of the three sources of confusion (point source, spectral intersection or arm confusion) has been identified. Note: the cleaning tables are calculated for all HEG and MEG orders and the unfiltered table includes a'warn' flag which indicates that the geometry of the star-pair was appropriate for confusion to occur but the respective threshold for confusion was not met. This usually means the confusion source did not have enough counts to cause confusion. 

### Let's clean the confused portions of the spectrum and responses using the CrissCross function clean_spec(). NOTE: This tool removes ALL counts in the confused regions and not just counts from the confusing source. However, CrissCross has many tunable parameters to allow some level of potential confusion through based on the SNR of the source events compared to the confused events in the confused bandpass. If you find CrissCross is unnecessarily removing counts then please re-run with lower confusion threshold parameters.

In [ ]:
help(clean_spec)

### clean_spec() will attempt to identify the relevant response files when provided with a PHA2 file. This is important because clean_spec also zeros out the ARF for each source of confusion. Clean spec creates a new PHA2 file and associated ARFs. It will not modify the original spectral file or responses.

In [ ]:
clean_spec(cc_table=cc_table_coup, pha_file = pha_data, spec_root=coup_root, arf_file=None, resp_dir=None)

### clean_spec produced a new PHA2 file and four new ARF files. So lets load those and take a look

In [ ]:
spec_root=f'{coup_root}_obsid_{obs}'
cleaned_pha2 = f'{spec_root}_cleaned.pha2'

cleaned_arf_meg_p1 = f'{spec_root}_meg+1_cleaned.arf'
cleaned_arf_meg_m1 = f'{spec_root}_meg-1_cleaned.arf'
cleaned_arf_heg_p1 = f'{spec_root}_heg+1_cleaned.arf'
cleaned_arf_heg_m1 = f'{spec_root}_heg-1_cleaned.arf'


### Let's run clean() in our session which will remove our previously loaded PHA2 file and responses.


In [ ]:
clean()

load_data(cleaned_pha2)

### We will find the responses again and load the ARFs and RMFs

In [ ]:
for i in range(0,12):
    if rmf_list_ordered[i] != 'no match':
        #load_arf(id=i+1, arg=arf_list_ordered[i])
        load_rmf(id=i+1, arg=rmf_list_ordered[i])
    else:
        delete_data(i+1) #must delete other datasets because I can't plot in wavelength unless ALL orders have responses...

In [ ]:
load_arf(3, cleaned_arf_heg_m1)
load_arf(4, cleaned_arf_heg_p1)
load_arf(9, cleaned_arf_meg_m1)
load_arf(10, cleaned_arf_meg_p1)

### Now we plot the cleaned spectra and ARFs

In [ ]:
spec_ids = [3,4,9,10]

set_analysis('wave') #view spectra in wavelength

bin_counts=3
for i in spec_ids:
    group_counts(i, bin_counts)

plot("data", 3, "data", 4, "data", 9, "data", 10)

In [ ]:
plot("arf", 3, "arf", 4, "arf", 9, "arf", 10)

### We can directly see the identified regions of confusion where the ARF goes to zero

### Save the spectral plot data for comparison of original vs cleaned

In [ ]:
spec_arr_cleaned = [None]*4
spec_ids = [3,4,9,10]

for i in range(len(spec_arr_cleaned)):
    spec_arr_cleaned[i] = get_data(id=spec_ids[i])

# Create a 2×2 figure (4 sub‑plots)
fig, axes = plt.subplots(
    nrows=2, ncols=2,                # 2 rows, 2 columns
    figsize=(10, 8),                # optional: figure size in inches
    sharex=True, sharey=True,      # share axes (optional)
)

ax = axes.ravel()

for i in range(len(spec_arr_cleaned)):
    ax[i].plot(spec_arr_cleaned[i].x, spec_arr_cleaned[i].y)
    ax[i].set_xlim(0,30)
    ax[i].set_xlabel('Wavelength')
    ax[i].set_ylabel('Counts s$^{-1}$ $A^{-1}$')
    ax[i].set_title(f'{coup_root} {hetg_info[i]}')    

fig.tight_layout(pad=1.0)
plt.show()

### Now lets compare the cleaned spectra (blue) to the original spectra (red)

In [ ]:
# Create a 2×2 figure (4 sub‑plots)
fig, axes = plt.subplots(
    nrows=2, ncols=2,                # 2 rows, 2 columns
    figsize=(10, 8),                # optional: figure size in inches
    sharex=True, sharey=True,      # share axes (optional)
)

ax = axes.ravel()

for i in range(len(spec_arr_cleaned)):
    ax[i].plot(spec_arr[i].x, spec_arr[i].y, color='red', label='confused')
    ax[i].plot(spec_arr_cleaned[i].x, spec_arr_cleaned[i].y, color='blue',label='cleaned')
    ax[i].set_xlim(0,30)
    ax[i].set_xlabel('Wavelength [A]')
    ax[i].set_ylabel('Counts s$^{-1}$ A$^{-1}$')
    ax[i].set_title(f'{coup_root} {hetg_info[i]}')
    ax[i].legend()

#ax.legend()
fig.tight_layout(pad=1.0)
plt.show()

### We can see several portions of each spectrum have been identified to have events associated with other sources in the field assigned to our extracted source (e.g., confusion). Notice what would have appeared as emission lines now are identfied as emission from a different source. The small ranges of confusion are typically associated with point source and spectral intersection whereas the large bandpass of confusion are typically the result of arm confusion (e.g., another bright source falling on the dispersed arm of the extract source.)

### We can double check the confusion tables to see specific instances of confusion and the source causing them.

In [ ]:
dmlist.punlearn
dmlist.infile = f'{cc_table_coup}[flag=confused, order=+1, grating_type=meg]'
dmlist.opt='data'
dmlist.rows='1:50'

dmlist()

### From the table, it appears that the MEG+1 wavelength range of ~8-9.5 A is affected by two sources of confusion. A 0th order point source (COUP 187) which has 695 0th order counts landed right on the dispersed spectrum and the dispersed spectrum of COUP 1390 (1618 0th order counts) also intersected with the extracted spectrum.

### Lets take a closer look at the events assigned to the HEG or MEG spectrum to visually assess how well CrissCross identified confusion. This method only works on evt2 files made for the extracted source of interest.

In [ ]:
#use pycrates to read in the event file and inspect the columns
from pycrates import read_file

evt2_file = f'{obs}/repro/{coup_root}_repro_evt2.fits'
evt_data = read_file(evt2_file)
evt_data.get_colnames()

### Save the relevant event values to new arrays for plotting purposes. HETG event files contain extra columns for events associated with the extracted spectrum.

In [ ]:
tg_lam = evt_data.tg_lam.values
tg_d = evt_data.rd.tg_d.values
tg_m = evt_data.tg_m.values
tg_mlam = evt_data.tg_mlam.values
energy = evt_data.energy.values

In [ ]:
def filter_evt_crate(evt_crate, order_val, arm_par, tgd_par):
    """
    Function for filtering a crate evt file to return the event rows that satisfy the HETG order, arm type (meg/heg) and tg_d value conditions. 
    """
    crate_elements = np.where( (evt_crate.tg_m.values == order_val) & (evt_crate.tg_part.values == arm_par) & (np.abs(evt_crate.rd.tg_d.values) < tgd_par) )

    return(crate_elements)

### We have to filter the event file to retrieve order- and arm-specific event information for plotting.

In [ ]:
#create the filters for plotting only the relevant orders and arm type
meg_p1 = filter_evt_crate(evt_data, 1, 2, 8E-3)
meg_m1 = filter_evt_crate(evt_data,-1, 2, 8E-3)
heg_p1 = filter_evt_crate(evt_data, 1, 1, 8E-3)
heg_m1 = filter_evt_crate(evt_data, -1, 1, 8E-3)

fig, ax = plt.subplots(
    nrows=1, ncols=2,                # 2 rows, 2 columns
    figsize=(16, 5),                # optional: figure size in inches
)

#MEG PLOT
ax[0].scatter(-1*tg_lam[meg_m1], tg_d[meg_m1], s=1, color='green')
ax[0].scatter(tg_lam[meg_p1], tg_d[meg_p1], s=1, color='blue')
ax[0].plot([-30,30], [6.4E-4, 6.4E-4], color='orange') #default extraction width for HETG spectra
ax[0].plot([-30,30], [-6.4E-4, -6.4E-4], color='orange')
ax[0].set_title('MEG -1 and +1')

#HEG PLOT
ax[1].scatter(-1*tg_lam[heg_m1], tg_d[heg_m1], s=1, color='green')
ax[1].scatter(tg_lam[heg_p1], tg_d[heg_p1], s=1, color='blue')
ax[1].set_title('HEG -1 and +1')
ax[1].plot([-15,15], [6.4E-4, 6.4E-4], color='orange')
ax[1].plot([-15,15], [-6.4E-4, -6.4E-4], color='orange')

for i in ax:
    i.set_xlabel('m * Wavelength [Angstrom]')
    i.set_ylabel('Cross Dispersion Angle [deg]')

### The above plot shows the MEG (left) and HEG (right) events associated with the extracted source. Specifically, the events within the orange box will be those that are included in the spectrum. The X-axis denotes the order (1 or -1) * wavelength and the y-axis denotes the cross dispersion angle (tg_d) in degrees. This y-axis value is essentially the counts above and below the 'line' of the disperssed spectrum.

### A single unconfused source should only show a 'bar' of events within the orange box. On the MEG+1 plot (left, blue) you can see a strong vertical line at about 9 Angstroms which is the location of a 0th order (point source) field star that happend to fall on the dispersed spectrum of our extract source. The vertical nature of this feature is due to the size of the PSF of that source and many of those 0th-order events were the appropriate energy (e.g., 9 A) to erroneously be assigned to the source of our extracted source. Other fainter 0th-order confusing sources can be seen in MEG-1 at approximately -11 and -5 A. The two large horizontal features shown in the MEG plot at y~0.008 and -0.006 represent the dispersed spectra of other stars in the field which are close to the extracted source. However, only events with a small range in cross dispersion angle (~+/- 6.3E-4) and assigned to the extracted source.

### The HEG -1 plot shows many examples of spectral-intersection confusion (e.g., the dispersed arm of another source overlapping the extracted source at the wavelength expected by ACIS order sorting). These are denoted with slanted lines at approximately -6.5 and -9 A. 

### This format of a plot is difficult to use to assess arm confusion so lets look at the data now with the y-axis as the ACIS-resolved CCD energy.

In [ ]:

#REDO the filtering with a smaller tg_d window to remove events outside default HETG extraction window
meg_p1 = filter_evt_crate(evt_data, 1, 2, 4.6E-4)
meg_m1 = filter_evt_crate(evt_data,-1, 2, 4.6E-4)
heg_p1 = filter_evt_crate(evt_data, 1, 1, 4.6E-4)
heg_m1 = filter_evt_crate(evt_data, -1, 1, 4.6E-4)

#display all events assigned to orders -3,-2,-1,+1,+2,+3
all_evts_meg_filt = np.where( (np.abs(evt_data.tg_m.values) <= 3) & (evt_data.tg_part.values == 2) )
all_evts_heg_filt = np.where( (np.abs(evt_data.tg_m.values) <= 3 ) & (evt_data.tg_part.values == 1) )


# Create a 2×2 figure (4 sub‑plots)
fig, ax = plt.subplots(
    nrows=1, ncols=2,                # 2 rows, 2 columns
    figsize=(16, 5),                # optional: figure size in inches
)

ax[0].scatter(tg_mlam[all_evts_meg_filt], energy[all_evts_meg_filt], s=1, color='grey')
ax[0].scatter(tg_mlam[meg_m1], energy[meg_m1], s=1, color='green')
ax[0].scatter(tg_mlam[meg_p1], energy[meg_p1], s=1, color='blue')
ax[0].set_title('MEG -1 and +1')


ax[1].scatter(tg_mlam[all_evts_heg_filt], energy[all_evts_heg_filt], s=1, color='grey')
ax[1].scatter(tg_mlam[heg_m1], energy[heg_m1], s=1, color='green')
ax[1].scatter(tg_mlam[heg_p1], energy[heg_p1], s=1, color='blue')
ax[1].set_title('HEG -1 and +1')

for i in ax:
    i.set_ylim(300,7500)
    i.set_xlabel('m * Wavelength [Angstrom]')
    i.set_ylabel('ACIS Energy [eV]')

### The above plots are the so-called HETG 'banana plots' and help reveal the extent to which a source's arm/order suffers from confusion. The HETG gratings equation tells us the wavelegnth of every event based on its position along the dispersion direction from the 0th order. This wavelength (energy) is then compared to it's ACIS CCD-determined energy to ensure it is compatible with the gratings-derived energy (wavelength). Only events whose ACIS CCD energy are consistent with the gratings-determined energy are ultimately assigned to the extracted spectrum. Shown here in gray are all events assigned to any of the -3, -2, -1, +1, +2, +3 orders. The green and blue events are those that fall within the 4.6E-3 cross dispersion direction denoted by the yellow bars in the previous plots. 

### Looking closer at the MEG+1 portion of the spectrum you can see the previously discussed 'point source confusion' at ~9 A. Note how it is a vertical bar in all three positive orders because the 0th order contains events over a much larger portion of ACIS CCD energy compared to the wavelength space at 9 A. Also note the curved feature starting at ~13 A in the MEG+3 order and bending ultimately towards the MEG+1 order at ~ 20 A. This is a clear indication of arm confusion from a source somewhere else on the detector but in line with the dispersed spectrum of our extracted source. The confusing source would likely contaminate all portions of the MEG+1 spectrum >~ 20